# Customer Churn Prediction — Model Evaluation

## Objective
The objective of this notebook is to evaluate and compare the machine-learning models developed for predicting customer churn.

## Models to Compare
- Logistic Regression
- Decision Tree

## Evaluation Metrics
- Accuracy
- Precision
- Recall
- F1-score
- Confusion Matrix

## Expected Outcome
Identify the most suitable model for predicting customer churn based on its evaluation results.

In [20]:

# Day 8: Model Evaluation and Prediction Pipeline

from pathlib import Path
import pandas as pd
import numpy as np
import joblib

print("Libraries imported successfully!")


Libraries imported successfully!


In [21]:

# Locate the project folder and saved models

current_path = Path.cwd()

possible_paths = [
    current_path,
    current_path.parent,
    current_path.parent.parent
]

project_folder = None

for path in possible_paths:
    if (path / "models").is_dir() and (path / "data").is_dir():
        project_folder = path
        break

if project_folder is None:
    raise FileNotFoundError(
        "Project folder not found. Open the notebook "
        "inside the Customer-Churn-Prediction project."
    )

models_folder = project_folder / "models"
data_folder = project_folder / "data"

print("Project folder:", project_folder)
print("\nSaved model files:")

model_files = list(models_folder.rglob("*.joblib"))

for file in model_files:
    print(file.relative_to(project_folder))

if not model_files:
    print("No .joblib files found in the models folder.")


Project folder: c:\Users\Dell\OneDrive\Desktop\Customer-Churn-Prediction

Saved model files:
models\decision_tree_model.joblib
models\logistic_regression_model.joblib


In [22]:

# Load the Day 7 model comparison results

results_path = project_folder / "model_comparison_results.csv"

if not results_path.exists():
    raise FileNotFoundError(
        "Day 7 comparison CSV not found at: "
        + str(results_path)
    )

comparison = pd.read_csv(results_path, index_col=0)

print("Day 7 comparison results:")
display(comparison.round(4))

if "F1-score" not in comparison.index:
    raise ValueError(
        "F1-score row not found. Check the Day 7 results CSV."
    )

best_model_name = comparison.loc["F1-score"].idxmax()

print("\nSelected model based on F1-score:")
print(best_model_name)


Day 7 comparison results:


,Logistic Regression,Decision Tree
Accuracy,0.8041,0.7268
Precision,0.6467,0.4861
Recall,0.5775,0.5134
F1-score,0.6102,0.4993



Selected model based on F1-score:
Logistic Regression


In [23]:

# Match the Day 7 winner to its saved model file

winner = str(best_model_name).strip().lower()

if "logistic" in winner:
    candidates = [
        f for f in model_files
        if "logistic" in f.stem.lower()
    ]

elif "decision" in winner and "tree" in winner:
    candidates = [
        f for f in model_files
        if "decision" in f.stem.lower()
        and "tree" in f.stem.lower()
    ]

elif "tree" in winner:
    candidates = [
        f for f in model_files
        if "tree" in f.stem.lower()
    ]

else:
    raise ValueError(
        f"Cannot identify the model type from: {best_model_name}"
    )

if len(candidates) == 0:
    raise FileNotFoundError(
        "No saved model file matches the Day 7 winner. "
        "Check the filenames printed in Step 3."
    )

if len(candidates) > 1:
    print("Multiple matching models found:")
    for file in candidates:
        print(file)
    raise ValueError(
        "Select the correct saved model, considering "
        "the original and tuned model results."
    )

best_model_path = candidates[0]

print("Selected model:", best_model_name)
print("Model file:", best_model_path)


Selected model: Logistic Regression
Model file: c:\Users\Dell\OneDrive\Desktop\Customer-Churn-Prediction\models\logistic_regression_model.joblib


In [24]:

# Load the selected model without retraining

best_model = joblib.load(best_model_path)

print("Selected model loaded successfully!")
print("Model type:", type(best_model).__name__)

if hasattr(best_model, "named_steps"):
    print("\nPipeline steps:")
    for step_name, step_object in best_model.named_steps.items():
        print(f"{step_name}: {type(step_object).__name__}")
else:
    print(
        "\nThis saved object is not a complete sklearn Pipeline."
    )
    print(
        "We must verify its preprocessing requirements "
        "before using it on new customer data."
    )


Selected model loaded successfully!
Model type: Pipeline

Pipeline steps:
preprocessor: ColumnTransformer
classifier: LogisticRegression


In [25]:

# Locate the existing X_test.csv file

test_file_candidates = list(
    data_folder.rglob("X_test.csv")
)

if len(test_file_candidates) == 0:
    raise FileNotFoundError(
        "X_test.csv was not found inside the data folder."
    )

if len(test_file_candidates) > 1:
    print("Multiple X_test.csv files found:")
    for file in test_file_candidates:
        print(file)
    raise ValueError(
        "Please identify the same X_test.csv used in Day 7."
    )

X_test_path = test_file_candidates[0]
X_test = pd.read_csv(X_test_path)

print("Test data loaded successfully!")
print("File:", X_test_path)
print("Rows and columns:", X_test.shape)
print("\nFeature columns:")
print(X_test.columns.tolist())


Test data loaded successfully!
File: c:\Users\Dell\OneDrive\Desktop\Customer-Churn-Prediction\data\processed\model_inputs\X_test.csv
Rows and columns: (1409, 20)

Feature columns:
['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'CLTV']


In [26]:

# Prepare one sample customer using the existing test features

if X_test.empty:
    raise ValueError("X_test.csv contains no customer rows.")

X_sample = X_test.iloc[[0]].copy()

# Replace cells containing only spaces with missing values
X_sample = X_sample.replace(r'^\s*$', np.nan, regex=True)

print("Sample customer prepared!")
print("Sample shape:", X_sample.shape)
display(X_sample)


Sample customer prepared!
Sample shape: (1, 20)


,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,CLTV
0,Male,No,Yes,No,72,Yes,Yes,Fiber optic,Yes,Yes,Yes,Yes,Yes,Yes,Two year,Yes,Credit card (automatic),114.05,8468.2,4842


In [27]:

# Predict churn for the sample customer

if not hasattr(best_model, "named_steps"):
    raise ValueError(
        "The selected model is not a complete preprocessing "
        "pipeline. Verify the original preprocessing first."
    )

sample_prediction = best_model.predict(X_sample)

print("Prediction completed successfully!")
print("Selected model:", best_model_name)
print("Predicted class:", sample_prediction[0])

if sample_prediction[0] == 1:
    print("Interpretation: Customer is predicted to churn.")
elif sample_prediction[0] == 0:
    print("Interpretation: Customer is predicted to stay.")
else:
    print(
        "Check the target-label mapping to interpret "
        "this predicted class."
    )


Prediction completed successfully!
Selected model: Logistic Regression
Predicted class: 0
Interpretation: Customer is predicted to stay.


In [28]:

# Save the sample prediction

prediction_output = pd.DataFrame({
    "sample_row_index": [X_sample.index[0]],
    "selected_model": [best_model_name],
    "predicted_class": [sample_prediction[0]]
})

prediction_path = (
    project_folder / "day8_sample_prediction.csv"
)

prediction_output.to_csv(prediction_path, index=False)

print("Prediction output saved successfully!")
print("File location:", prediction_path)

display(prediction_output)


Prediction output saved successfully!
File location: c:\Users\Dell\OneDrive\Desktop\Customer-Churn-Prediction\day8_sample_prediction.csv


,sample_row_index,selected_model,predicted_class
0,0,Logistic Regression,0


# Day 8: Model Evaluation and Prediction Pipeline

## Objective
To select the best-performing model using Day 7 results, verify its preprocessing pipeline, and test a customer churn prediction.

## Tasks Completed
1. Reviewed the Day 7 model comparison results.
2. Identified the selected model based on F1-score.
3. Loaded the existing saved model without retraining it.
4. Checked whether preprocessing steps were included in the model pipeline.
5. Prepared a sample customer using the existing test features.
6. Generated a churn prediction using the selected model.
7. Saved the sample prediction to a CSV file.

## Conclusion
The selected model was evaluated using the Day 7 comparison results. Its prediction pipeline was tested using a sample customer record. The model can be prepared for future customer predictions after verifying that the required preprocessing and feature format are consistent with training.